In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from loguru import logger


Part 1 : We look for molecules with the same id but in different json files

In [262]:
# We load the two main files :
# - entities.tsv: contains the extracted entities
# - grounded_molecules.tsv: conyain the results of the molecules grounding
# - (molecule name,type of database asked, the grounded id etc...)
entities = pd.read_csv("../data/entities.tsv", sep="\t")
grounded = pd.read_csv("../results/ground_molecule/grounded_molecules.tsv", sep="\t")

In [263]:
entities

,entity,category,json_file
0,popc,MOL,zenodo_34415.json
1,amber,FFM,zenodo_34415.json
2,lipid14,FFM,zenodo_34415.json
3,cacl2,MOL,zenodo_34415.json
4,popc,MOL,zenodo_34415.json
...,...,...,...
1796,peggo,MOL,figshare_11894358.json
1797,peg,MOL,figshare_11894358.json
1798,dox,MOL,figshare_11894358.json
1799,dox,MOL,figshare_11894358.json


In [264]:
grounded

,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
2,ca,CHEBI,No errors,22984,22.602823,calcium atom,100
3,cl,CHEBI,No errors,17996,26.846502,chloride,10884
4,dmpc,CHEBI,No errors,241349,39.092686,dimyristoyl phosphatidylcholine,2
...,...,...,...,...,...,...,...
417,organic structure-directing agents,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
418,silicalite‑1,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
419,silicalite-1,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
420,osdas,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available


In [265]:
# We only select the entities that are molecules
mol_entities = entities[entities["category"] == "MOL"]

In [266]:
mol_entities

,entity,category,json_file
0,popc,MOL,zenodo_34415.json
3,cacl2,MOL,zenodo_34415.json
4,popc,MOL,zenodo_34415.json
5,cacl2,MOL,zenodo_34415.json
11,popc,MOL,zenodo_34415.json
...,...,...,...
1796,peggo,MOL,figshare_11894358.json
1797,peg,MOL,figshare_11894358.json
1798,dox,MOL,figshare_11894358.json
1799,dox,MOL,figshare_11894358.json


In [267]:
# We merge the two dataframes on the name of molecules
merged = mol_entities.merge(grounded, left_on="entity", right_on="MOL")

In [268]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
2,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
3,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
4,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
...,...,...,...,...,...,...,...,...,...,...
1122,peggo,MOL,figshare_11894358.json,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
1123,peg,MOL,figshare_11894358.json,peg,CHEBI,No errors,46793,31.78609,poly(ethylene glycol),23
1124,dox,MOL,figshare_11894358.json,dox,CHEBI,No errors,131529,24.0,pyridoxal hydrochloride,409
1125,dox,MOL,figshare_11894358.json,dox,CHEBI,No errors,131529,24.0,pyridoxal hydrochloride,409


In [269]:
# We only work with molecules were the grounding was succesfull
merged = merged[merged["MOL_ID"] != "Not Available"]

In [270]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
2,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
3,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
4,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
...,...,...,...,...,...,...,...,...,...,...
1120,go,MOL,figshare_11894358.json,go,CHEBI,No errors,17720,9.268637,"2,3-bisphospho-D-glyceric acid",25
1121,dox,MOL,figshare_11894358.json,dox,CHEBI,No errors,131529,24.0,pyridoxal hydrochloride,409
1123,peg,MOL,figshare_11894358.json,peg,CHEBI,No errors,46793,31.78609,poly(ethylene glycol),23
1124,dox,MOL,figshare_11894358.json,dox,CHEBI,No errors,131529,24.0,pyridoxal hydrochloride,409


In [ ]:
# We delete every duplicated row based on :
# - the mol id and type, the mol name and the json file
merged = merged.drop_duplicates(subset=["MOL_ID", "MOL_TYPE", "entity", "json_file"])


In [272]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
5,ca,MOL,zenodo_34415.json,ca,CHEBI,No errors,22984,22.602823,calcium atom,100
6,cl,MOL,zenodo_34415.json,cl,CHEBI,No errors,17996,26.846502,chloride,10884
7,dmpc,MOL,figshare_8046437.json,dmpc,CHEBI,No errors,241349,39.092686,dimyristoyl phosphatidylcholine,2
...,...,...,...,...,...,...,...,...,...,...
1083,peg,MOL,figshare_11894358.json,peg,CHEBI,No errors,46793,31.78609,poly(ethylene glycol),23
1084,graphene oxide,MOL,figshare_11894358.json,graphene oxide,CHEBI,No errors,132889,52.5388,graphene oxide,1
1086,dox,MOL,figshare_11894358.json,dox,CHEBI,No errors,131529,24.0,pyridoxal hydrochloride,409
1088,go,MOL,figshare_11894358.json,go,CHEBI,No errors,17720,9.268637,"2,3-bisphospho-D-glyceric acid",25


In [ ]:
# We want to count in how many json file we find the same grounding
# - We start by grouping the molecule based on their ID and TYPE
# - We apply the count method on the json_file column
#   to count in how many different json file we can find this grounding

counts = merged.groupby(["MOL_ID", "MOL_TYPE"])["json_file"].nunique()

In [15]:
counts

MOL_ID    MOL_TYPE
10043     CHEBI       1
10550     CHEBI       2
10674918  PubChem     2
12834     CHEBI       1
131439    CHEBI       1
                     ..
91189     CHEBI       1
91219     CHEBI       1
94310     CHEBI       1
95008     CHEBI       1
99479     PubChem     1
Name: json_file, Length: 259, dtype: int64

In [ ]:
# We reindex the columns so that MOL_ID and MOL_TYPE dont'appear as indexes
counts = counts.reset_index()

In [ ]:
# We rename the last column (number of json file wher we find the same grounding)
counts.columns = ["MOL_ID", "MOL_TYPE", "nb_json_files"]

In [18]:
counts

,MOL_ID,MOL_TYPE,nb_json_files
0,10043,CHEBI,1
1,10550,CHEBI,2
2,10674918,PubChem,2
3,12834,CHEBI,1
4,131439,CHEBI,1
...,...,...,...
254,91189,CHEBI,1
255,91219,CHEBI,1
256,94310,CHEBI,1
257,95008,CHEBI,1


In [ ]:
# We identify the grounding that have been found in more then one json_file
duplicates = counts[counts["nb_json_files"] > 1]

In [20]:
duplicates

,MOL_ID,MOL_TYPE,nb_json_files
1,10550,CHEBI,2
2,10674918,PubChem,2
7,132889,CHEBI,2
12,133805,CHEBI,2
36,16113,CHEBI,10
39,16183,CHEBI,2
40,16199,CHEBI,2
59,17761,CHEBI,2
65,17996,CHEBI,8
94,241349,CHEBI,4


In [ ]:
# We merge the duplicated data frame that contains the MOL_ID and MOL_TYPE that have
# been found in different json files with the merge dtafarame that contains all the
# information about the molecule grounded.
# We want to retreive the information about the molecule that have been found
# in different json file
details = merged.merge(duplicates[["MOL_ID", "MOL_TYPE"]], on=["MOL_ID", "MOL_TYPE"])

In [279]:
details

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.635445,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.092686,calcium dichloride,1
2,cl,MOL,zenodo_34415.json,cl,CHEBI,No errors,17996,26.846502,chloride,10884
3,dmpc,MOL,figshare_8046437.json,dmpc,CHEBI,No errors,241349,39.092686,dimyristoyl phosphatidylcholine,2
4,dppc,MOL,zenodo_1009027.json,dppc,CHEBI,No errors,40265,40.72799,"1,2-di-O-palmitoyl-sn-glycero-3-phosphocholine",1
...,...,...,...,...,...,...,...,...,...,...
133,dopc,MOL,zenodo_573274.json,dopc,CHEBI,No errors,52360,40.72799,"1,2-dioleoyl-sn-glycero-3-phosphocholine(1+)",1
134,dlpc,MOL,zenodo_573274.json,dlpc,CHEBI,No errors,60273,37.08416,"1,2-dilauroyl-sn-glycero-3-phosphocholine(1+)",2
135,nacl,MOL,zenodo_573274.json,nacl,CHEBI,No errors,26710,40.72799,sodium chloride,1
136,nacl,MOL,figshare_4508975.json,nacl,CHEBI,No errors,26710,40.72799,sodium chloride,1


In [ ]:
# We only kepp intersting columns such as :
# - the mol_id the mol_type the json_file and the mol name (entity)
details = details[["MOL_ID", "MOL_TYPE", "json_file", "entity"]]

In [24]:
details

,MOL_ID,MOL_TYPE,json_file,entity
0,73001,CHEBI,zenodo_34415.json,popc
1,3312,CHEBI,zenodo_34415.json,cacl2
2,17996,CHEBI,zenodo_34415.json,cl
3,241349,CHEBI,figshare_8046437.json,dmpc
4,40265,CHEBI,zenodo_1009027.json,dppc
...,...,...,...,...
133,52360,CHEBI,zenodo_573274.json,dopc
134,60273,CHEBI,zenodo_573274.json,dlpc
135,26710,CHEBI,zenodo_573274.json,nacl
136,26710,CHEBI,figshare_4508975.json,nacl


In [ ]:
# We sort the data frame for a better readability
details = details.sort_values(["MOL_ID", "MOL_TYPE", "json_file"])

In [26]:
details

,MOL_ID,MOL_TYPE,json_file,entity
95,10550,CHEBI,figshare_20485017.json,entf*
73,10550,CHEBI,figshare_20485059.json,entf*
13,10674918,PubChem,zenodo_51750.json,dmtap
52,10674918,PubChem,zenodo_53212.json,dmtap
137,132889,CHEBI,figshare_11894358.json,graphene oxide
...,...,...,...,...
119,73001,CHEBI,zenodo_5362218.json,popc
104,73001,CHEBI,zenodo_6010416.json,popc
112,73001,CHEBI,zenodo_6817824.json,popc
75,73001,CHEBI,zenodo_6988344.json,popc


In [ ]:
# We save the dataframe in a tsv file
details.to_csv("../results/same_grounding_molecules.tsv", sep="\t", index=False)

Part 2 : We create the knowledge graphs (grounded and ungroundes) for the molecules previously identied

In [ ]:
# We define a list of ions that we awant to exclude from our graph
IONS = {
    "na",
    "na+",
    "cl",
    "cl-",
    "nacl",
}

In [ ]:
# We create a function to format the json_file name :
# - exemple: zenodo_123456 to zenodo\n123456
def format_dataset(json_file: str) -> str:
    json_file = json_file.strip(".json")
    if "zenodo_" in json_file:
        json_file = json_file.replace("zenodo_", "zenodo\n")
    if "figshare_" in json_file:
        json_file = json_file.replace("figshare_", "figshare\n")
    return json_file


In [ ]:
# We create a list containning the json_file (with the previous function)
json_nodes = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        json_nodes.append(format_dataset(row["json_file"]))

In [201]:
json_nodes

['figshare\n20485017',
 'figshare\n20485059',
 'zenodo\n51750',
 'zenodo\n53212',
 'figshare\n11894358',
 'figshare\n13836577',
 'figshare\n11704443',
 'zenodo\n4012224',
 'figshare\n14994624',
 'figshare\n4806544',
 'zenodo\n247386',
 'zenodo\n259443',
 'zenodo\n2645909',
 'zenodo\n2653735',
 'zenodo\n3988469',
 'zenodo\n4445375',
 'zenodo\n6010416',
 'zenodo\n6144286',
 'zenodo\n6144286',
 'figshare\n12661589',
 'zenodo\n4106413',
 'figshare\n12661589',
 'zenodo\n6791876',
 'zenodo\n1219494',
 'zenodo\n3228177',
 'figshare\n14511885',
 'figshare\n14994624',
 'figshare\n8046437',
 'zenodo\n51750',
 'zenodo\n53212',
 'zenodo\n1488094',
 'zenodo\n44622',
 'figshare\n14511885',
 'zenodo\n1198454',
 'figshare\n22203473',
 'zenodo\n3592499',
 'zenodo\n573274',
 'figshare\n11808396',
 'zenodo\n3975394',
 'figshare\n14994624',
 'figshare\n14994624',
 'figshare\n3426170',
 'zenodo\n259443',
 'zenodo\n34415',
 'zenodo\n3613573',
 'zenodo\n51185',
 'zenodo\n3613573',
 'zenodo\n6817824',
 'figsh

In [ ]:
# We create two list one for the molecule name and one for the molecule IDS
# - We check every row to make sure that the molecule is not and ions
mol_nodes = []
mol_id_nodes = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        mol_nodes.append(row["entity"])
        mol_id_nodes.append(row["MOL_TYPE"] + "\n" + row["MOL_ID"])

In [203]:
mol_nodes

['entf*',
 'entf*',
 'dmtap',
 'dmtap',
 'graphene oxide',
 'graphene oxide',
 'md-2',
 'tmds',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterols',
 'methane',
 'methane',
 'urea',
 'urea',
 'ceramide',
 'ceramide',
 'chloride',
 'dmpc',
 'dmpc',
 'dmpc',
 'dmpc',
 'k+',
 'potassium',
 'sodium',
 'sodium',
 'pc',
 'depc',
 'depc',
 'lopinavir',
 'lopinavir',
 'tio2',
 'titanium dioxide',
 'tio2',
 'cacl_2',
 'cacl2',
 'cacl_2',
 'cacl_2',
 'popg',
 'popg',
 'graphene',
 'graphene',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 '1,2-dimyristoyl-sn-glycero-3-phosphocholine',
 'dimyristoylphosphatidylcholine',
 'dimyristoylphosphatidylcholine',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dlpc',
 'dlpc',
 'pope',
 'pope',
 'pope',
 'pops',
 'phosphatidylcholine',
 'phosphatidylcholine',
 'amyloid-β',
 'amyloid 

In [ ]:
# We create edges between the molecule names and the json file it has been found in.
dataset_mol_edges = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        dataset_mol_edges.append({format_dataset(row["json_file"]), row["entity"]})

In [ ]:
# We create edges between the molecule names and the molecule id.
mol_molids_edges = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        mol_molids_edges.append({row["MOL_TYPE"] + "\n" + row["MOL_ID"], row["entity"]})

In [ ]:
# We create an ungrounded knowledge graph that has:
# - Nodes: the json_file name and the molecule name
# - edges: link between the json file and its molecules
knowledge_graph = nx.Graph()

knowledge_graph.add_nodes_from(json_nodes, color="#f8ed62")
knowledge_graph.add_nodes_from(mol_nodes, color="skyblue")

knowledge_graph.add_edges_from(dataset_mol_edges)

In [207]:
mol_nodes


['entf*',
 'entf*',
 'dmtap',
 'dmtap',
 'graphene oxide',
 'graphene oxide',
 'md-2',
 'tmds',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterols',
 'methane',
 'methane',
 'urea',
 'urea',
 'ceramide',
 'ceramide',
 'chloride',
 'dmpc',
 'dmpc',
 'dmpc',
 'dmpc',
 'k+',
 'potassium',
 'sodium',
 'sodium',
 'pc',
 'depc',
 'depc',
 'lopinavir',
 'lopinavir',
 'tio2',
 'titanium dioxide',
 'tio2',
 'cacl_2',
 'cacl2',
 'cacl_2',
 'cacl_2',
 'popg',
 'popg',
 'graphene',
 'graphene',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 '1,2-dimyristoyl-sn-glycero-3-phosphocholine',
 'dimyristoylphosphatidylcholine',
 'dimyristoylphosphatidylcholine',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dlpc',
 'dlpc',
 'pope',
 'pope',
 'pope',
 'pops',
 'phosphatidylcholine',
 'phosphatidylcholine',
 'amyloid-β',
 'amyloid 

In [ ]:
# We create a grounded knowledge graph that has:
# - Nodes: - the json_file name
#          - the molecule names
#          - the molecule ids
# - edges: - link between the json file and its molecules
#          - link between the molecule and its grounding id
knowledge_graph_grounded = nx.Graph()

knowledge_graph_grounded.add_nodes_from(json_nodes, color="#f8ed62")
knowledge_graph_grounded.add_nodes_from(mol_nodes, color="skyblue")
knowledge_graph_grounded.add_nodes_from(mol_id_nodes, color="#ffbaba")

knowledge_graph_grounded.add_edges_from(dataset_mol_edges)
knowledge_graph_grounded.add_edges_from(mol_molids_edges)
grounding_edges = list(mol_molids_edges)


In [209]:
dataset_mol_edges

[{'entf*', 'figshare\n20485017'},
 {'entf*', 'figshare\n20485059'},
 {'dmtap', 'zenodo\n51750'},
 {'dmtap', 'zenodo\n53212'},
 {'figshare\n11894358', 'graphene oxide'},
 {'figshare\n13836577', 'graphene oxide'},
 {'figshare\n11704443', 'md-2'},
 {'tmds', 'zenodo\n4012224'},
 {'cholesterol', 'figshare\n14994624'},
 {'cholesterol', 'figshare\n4806544'},
 {'cholesterol', 'zenodo\n247386'},
 {'cholesterol', 'zenodo\n259443'},
 {'cholesterol', 'zenodo\n2645909'},
 {'cholesterol', 'zenodo\n2653735'},
 {'cholesterol', 'zenodo\n3988469'},
 {'cholesterol', 'zenodo\n4445375'},
 {'cholesterol', 'zenodo\n6010416'},
 {'cholesterol', 'zenodo\n6144286'},
 {'cholesterols', 'zenodo\n6144286'},
 {'figshare\n12661589', 'methane'},
 {'methane', 'zenodo\n4106413'},
 {'figshare\n12661589', 'urea'},
 {'urea', 'zenodo\n6791876'},
 {'ceramide', 'zenodo\n1219494'},
 {'ceramide', 'zenodo\n3228177'},
 {'chloride', 'figshare\n14511885'},
 {'dmpc', 'figshare\n14994624'},
 {'dmpc', 'figshare\n8046437'},
 {'dmpc', 'z

In [210]:
mol_molids_edges

[{'CHEBI\n10550', 'entf*'},
 {'CHEBI\n10550', 'entf*'},
 {'PubChem\n10674918', 'dmtap'},
 {'PubChem\n10674918', 'dmtap'},
 {'CHEBI\n132889', 'graphene oxide'},
 {'CHEBI\n132889', 'graphene oxide'},
 {'CHEBI\n133805', 'md-2'},
 {'CHEBI\n133805', 'tmds'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterols'},
 {'CHEBI\n16183', 'methane'},
 {'CHEBI\n16183', 'methane'},
 {'CHEBI\n16199', 'urea'},
 {'CHEBI\n16199', 'urea'},
 {'CHEBI\n17761', 'ceramide'},
 {'CHEBI\n17761', 'ceramide'},
 {'CHEBI\n17996', 'chloride'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n26216', 'k+'},
 {'CHEBI\n26216', 'potassiu

In [ ]:
# We create a function that will relabel long molecule names
# - exemple: 1,2-dimyristoyl-sn-glycero-3-phosphocholine -> 1,2-di...
def get_molecule_label(nodes):
    d = {}
    for n in nodes:
        d[n] = n if len(n) <= 10 else n[:10] + "..."
    return d


In [ ]:
# We use a visualisation function that will display the knowledge graphes
def visualize_graph(G, mol_nodes, out, red_edges=None):

    node_colors = [G.nodes[n].get("color", "skyblue") for n in G.nodes]

    label_map = get_molecule_label(mol_nodes)
    labels = {n: label_map.get(n, n) for n in G.nodes}

    pos = nx.spring_layout(G, k=0.22)

    plt.figure(figsize=(22, 18))

    nx.draw(
        G,
        pos,
        labels=labels,
        node_color=node_colors,
        node_size=1000,
        font_size=10,
        edge_color="gray",
        linewidths=4,
        alpha=1,
    )
    # This part will allow us to highligth in red the new link made by the grounding
    if red_edges:
        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=(red_edges),
            edge_color="red",
            width=3,
        )

    plt.title("Knowledge Graph")
    plt.savefig(out)
    plt.close()

    logger.info("Saved:", out)

In [ ]:
# We create a list containing the list of the new edges created with the grounding
new_edges = (knowledge_graph_grounded.edges()) - (knowledge_graph.edges())

In [261]:
visualize_graph(knowledge_graph, mol_nodes, "non_grounded.png", red_edges=None)

visualize_graph(
    knowledge_graph_grounded, mol_nodes, "grounded.png", red_edges=new_edges
)

Saved: non_grounded.png
Saved: grounded.png


In [ ]:
# We create a function that will print the statistics of the graphes
def print_graph_stats(knowledge_graph: nx.Graph) -> None:
    """Print basic statistics about the graph."""
    num_components = nx.number_connected_components(knowledge_graph)
    logger.info(f"Number of nodes: {knowledge_graph.number_of_nodes()}")
    logger.info(f"Density: {nx.density(knowledge_graph)}")
    logger.info(f"Number of edges: {knowledge_graph.number_of_edges()}")
    logger.info(f"Number of disjoint subgraphs: {num_components}")
    logger.info("CONNECTED COMPONENT:")
    for i, component in enumerate(list(nx.connected_components(knowledge_graph))):
        logger.info(f"Connected components {i + 1}:{component}")


In [254]:
print_graph_stats(knowledge_graph)

2026-05-06 15:18:52.938 | INFO     | __main__:print_graph_stats:7 - Number of nodes: 94
2026-05-06 15:18:52.939 | INFO     | __main__:print_graph_stats:8 - Density: 0.024250743536948068
2026-05-06 15:18:52.939 | INFO     | __main__:print_graph_stats:9 - Number of edges: 106
2026-05-06 15:18:52.939 | INFO     | __main__:print_graph_stats:10 - Number of disjoint subgraphs: 8
2026-05-06 15:18:52.941 | INFO     | __main__:print_graph_stats:11 - CONNECTED COMPONENT:
2026-05-06 15:18:52.942 | INFO     | __main__:print_graph_stats:13 - Connected components 1:{'entf*', 'figshare\n20485017', 'figshare\n20485059'}
2026-05-06 15:18:52.942 | INFO     | __main__:print_graph_stats:13 - Connected components 2:{'dmtap', 'zenodo\n13814', 'dimyristoylphosphatidylcholine', 'zenodo\n1009027', 'zenodo\n1198454', 'dlpc', 'zenodo\n13853', 'zenodo\n1009607', 'zenodo\n34415', 'depc', 'popc', 'zenodo\n6988344', 'figshare\n14511885', 'zenodo\n51750', 'tio2', 'zenodo\n247386', 'figshare\n4806544', 'cholesterol', 

In [217]:
print_graph_stats(knowledge_graph_grounded)

2026-05-06 15:12:16.114 | INFO     | __main__:print_graph_stats:7 - Number of nodes: 120
2026-05-06 15:12:16.116 | INFO     | __main__:print_graph_stats:8 - Density: 0.019747899159663865
2026-05-06 15:12:16.118 | INFO     | __main__:print_graph_stats:9 - Number of edges: 141
2026-05-06 15:12:16.120 | INFO     | __main__:print_graph_stats:10 - Number of disjoint subgraphs: 6
2026-05-06 15:12:16.122 | INFO     | __main__:print_graph_stats:11 - CONNECTED COMPONENT:
2026-05-06 15:12:16.123 | INFO     | __main__:print_graph_stats:13 - Connected components 1:{'entf*', 'figshare\n20485017', 'CHEBI\n10550', 'figshare\n20485059'}
2026-05-06 15:12:16.124 | INFO     | __main__:print_graph_stats:13 - Connected components 2:{'zenodo\n1009027', 'zenodo\n1009607', 'zenodo\n34415', 'popc', 'zenodo\n6988344', 'CHEBI\n52360', 'CHEBI\n241349', 'zenodo\n51185', 'CHEBI\n26708', 'popg', 'zenodo\n30904', 'zenodo\n14591', 'pops', 'CHEBI\n17996', 'zenodo\n1293813', 'zenodo\n1118682', 'zenodo\n5362218', 'CHEBI\